In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [2]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

model_name = "/data/cuiluyi/open-r1/data/Qwen2.5-1.5B-Open-R1-GRPO"

/data/cuiluyi/anaconda3/envs/openr1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 02-22 23:52:51 __init__.py:183] Automatically detected platform cuda.


2025-02-22 23:52:52,025	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
# ===================== vLLM 初始化 =====================
# 注意：vLLM 会自动处理 device_map，无需手动指定设备
llm = LLM(
    model=model_name,
    tensor_parallel_size=1,       # 多GPU时调整
    dtype="auto",                 # 自动选择精度
    trust_remote_code=True        # Qwen需要此参数
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

INFO 02-22 23:53:02 config.py:526] This model supports multiple tasks: {'embed', 'score', 'reward', 'classify', 'generate'}. Defaulting to 'generate'.
INFO 02-22 23:53:02 llm_engine.py:232] Initializing a V0 LLM engine (v0.7.1) with config: model='/data/cuiluyi/open-r1/data/Qwen2.5-1.5B-Open-R1-GRPO', speculative_config=None, tokenizer='/data/cuiluyi/open-r1/data/Qwen2.5-1.5B-Open-R1-GRPO', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.18it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.17it/s]



INFO 02-22 23:53:06 model_runner.py:1116] Loading model weights took 2.8875 GB
INFO 02-22 23:53:08 worker.py:266] Memory profiling takes 1.91 seconds
INFO 02-22 23:53:08 worker.py:266] the current vLLM instance can use total_gpu_memory (23.68GiB) x gpu_memory_utilization (0.90) = 21.32GiB
INFO 02-22 23:53:08 worker.py:266] model weights take 2.89GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 2.02GiB; the rest of the memory reserved for KV Cache is 16.35GiB.
INFO 02-22 23:53:08 executor_base.py:108] # CUDA blocks: 38262, # CPU blocks: 9362
INFO 02-22 23:53:08 executor_base.py:113] Maximum concurrency for 32768 tokens per request: 18.68x
INFO 02-22 23:53:14 model_runner.py:1435] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_u

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:17<00:00,  2.04it/s]

INFO 02-22 23:53:31 model_runner.py:1563] Graph capturing finished in 17 secs, took 0.20 GiB
INFO 02-22 23:53:31 llm_engine.py:429] init engine (profile, create kv cache, warmup model) took 24.92 seconds


In [7]:
# ===================== 输入处理 =====================
# prompt = "Find the value of $x$ that satisfies the equation $4x+5 = 6x+7$."
prompt = "What is $10.0000198\\cdot 5.9999985401\\cdot 6.9999852$ to the nearest whole number?"

messages = [
    {"role": "system", "content": "Please reason step by step..."},
    {"role": "user", "content": prompt}
]

# 应用聊天模板（vLLM 0.4.0+ 支持自动模板，但这里显式处理）
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# ===================== vLLM 生成参数 =====================
# 定义停止条件（vLLM 原生支持字符串停止词）
stop_tokens = ["\n\n"]  # 直接使用字符串而非token ID

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.95,
    top_k=-1,
    max_tokens=2048,
    # stop=stop_tokens,            # 直接指定停止字符串
    n=1,                         # 生成3个结果（相当于num_return_sequences）
    include_stop_str_in_output=True,
)

# ===================== 执行生成 =====================
outputs = llm.generate([text], sampling_params)

# ===================== 结果处理 =====================
for i, output in enumerate(outputs[0].outputs):  # 提取第一个prompt的结果
    print(f"\\section{{step {i + 1}}}\n")
    print(output.text)

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it, est. speed input: 47.99 toks/s, output: 160.69 toks/s]

\section{step 1}

<think> To find the value of \(10.0000198 \cdot 5.9999985401 \cdot 6.9999852\) to the nearest whole number, we can use the fact that when multiplying two numbers, the value of the product is approximately equal to the square root of the cube of the first number times the cube of the second number. </think>

<think> The product of \(10.0000198\) and \(5.9999985401\) is approximately \(3.5000009\). When multiplied by \(6.9999852\), the result is approximately \(4.3999663\). </think>

<think> So, the product is approximately \(4.4000000\). The nearest whole number to this value is \(4\). </think>

<answer> \(4\)</answer>


In [ ]:
for item in outputs[0].outputs:
    print(repr(item.text))